In [4]:

# ============================================================
# TASK 2 — NUMPY ENGAGEMENT SCORE (WITHOUT LOOPS)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# Dataset paths
DATA_DIR = Path(Path.cwd())
US_DATA_PATH = DATA_DIR / "data/USvideos.csv"
GB_DATA_PATH = DATA_DIR / "data/GBvideos.csv"
IN_DATA_PATH = DATA_DIR / "data/INvideos.csv"

# Load datasets independently
us_df = pd.read_csv(US_DATA_PATH)
gb_df = pd.read_csv(GB_DATA_PATH)
in_df = pd.read_csv(IN_DATA_PATH)

# Add country information
us_df["country"] = "US"
gb_df["country"] = "GB"
in_df["country"] = "IN"

# Combine datasets
df = pd.concat(
    [us_df, gb_df, in_df],
    ignore_index=True
)

# Create NumPy arrays
views = df["views"].to_numpy(dtype=np.int64)
likes = df["likes"].to_numpy(dtype=np.int64)
dislikes = df["dislikes"].to_numpy(dtype=np.int64)
comment_count = df["comment_count"].to_numpy(dtype=np.int64)

print("Total records:", len(df))
print("Views shape:", views.shape)
print("Likes shape:", likes.shape)

# ------------------------------------------------------------
# 1. CALCULATE LIKE RATIO
# ------------------------------------------------------------

like_denominator = likes + dislikes

like_ratio = np.divide(
    likes,
    like_denominator,
    out=np.full(likes.shape, np.nan, dtype=float),
    where=(
        np.isfinite(like_denominator) &
        (like_denominator != 0)
    )
)

print("Like ratio:")
print(like_ratio[:10])


# ------------------------------------------------------------
# 2. CALCULATE COMMENT RATE
# ------------------------------------------------------------

comment_rate = np.divide(
    comment_count,
    views,
    out=np.full(views.shape, np.nan, dtype=float),
    where=(
        np.isfinite(views) &
        (views != 0)
    )
)

print("\nComment rate:")
print(comment_rate[:10])


# ------------------------------------------------------------
# 3. MIN-MAX NORMALIZATION FUNCTION
# ------------------------------------------------------------

def min_max_normalize(array):

    minimum = np.nanmin(array)
    maximum = np.nanmax(array)

    normalized = np.divide(
        array - minimum,
        maximum - minimum,
        out=np.full(array.shape, np.nan, dtype=float),
        where=(maximum != minimum) & np.isfinite(array)
    )

    # If all valid values are identical, assign zero
    # to valid observations.
    if maximum == minimum:
        normalized = np.where(
            np.isfinite(array),
            0.0,
            np.nan
        )

    return normalized


# Normalize individual metrics.

views_normalized = min_max_normalize(views)

likes_normalized = min_max_normalize(likes)

comments_normalized = min_max_normalize(comment_count)

print("\nNormalized views:")
print(views_normalized[:10])

print("\nNormalized likes:")
print(likes_normalized[:10])

print("\nNormalized comments:")
print(comments_normalized[:10])


# ------------------------------------------------------------
# 4. CALCULATE ENGAGEMENT SCORE
# ------------------------------------------------------------

engagement_score = (
    0.30 * views_normalized +
    0.40 * likes_normalized +
    0.30 * comments_normalized
)

print("\nEngagement scores:")
print(engagement_score[:10])


# ------------------------------------------------------------
# 5. LOG-SCALED VIEW COUNT
# ------------------------------------------------------------

views_log = np.log1p(views)

print("\nLog-scaled views:")
print(views_log[:10])


# ------------------------------------------------------------
# 6. IDENTIFY TOP 10% VIDEOS BY ENGAGEMENT
# ------------------------------------------------------------

valid_scores = engagement_score[
    np.isfinite(engagement_score)
]

if valid_scores.size == 0:
    raise ValueError("No valid engagement scores available.")

threshold = np.percentile(valid_scores, 90)

top_10_mask = (
    np.isfinite(engagement_score) &
    (engagement_score >= threshold)
)

print("\n90th percentile engagement threshold:")
print(threshold)

print("\nTop 10% mask:")
print(top_10_mask[:20])

print("\nNumber of top engagement records:")
print(np.sum(top_10_mask))

print("\nPercentage selected:")
print(
    round(
        np.sum(top_10_mask) /
        np.sum(np.isfinite(engagement_score)) * 100,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 7. CREATE A RESULT DATAFRAME FOR INSPECTION
# ------------------------------------------------------------

engagement_results = pd.DataFrame({
    "views": views,
    "likes": likes,
    "dislikes": dislikes,
    "comment_count": comment_count,
    "like_ratio": like_ratio,
    "comment_rate": comment_rate,
    "views_normalized": views_normalized,
    "likes_normalized": likes_normalized,
    "comments_normalized": comments_normalized,
    "engagement_score": engagement_score,
    "views_log": views_log,
    "top_10_percent": top_10_mask
})

print("\nEngagement results:")
print(engagement_results.head(10))

print("\nTop 10% engagement records:")
print(
    engagement_results[top_10_mask]
    .sort_values("engagement_score", ascending=False)
    .head(20)
)


Total records: 46230
Views shape: (46230,)
Likes shape: (46230,)
Like ratio:
[0.92401857 0.70199879 0.90703465 0.95652174 0.98544282 0.95370136
 0.96099928 0.87250937 0.970141   0.5759457 ]

Comment rate:
[0.00280095 0.00728886 0.00186483 0.00235061 0.00238771 0.00338812
 0.00199492 0.00424703 0.00208841 0.00627889]

Normalized views:
[1.31636320e-04 5.46955021e-04 6.39179999e-04 1.94506529e-05
 1.03142236e-03 1.42141668e-03 7.91838658e-04 7.87287699e-03
 2.35958013e-04 1.43446694e-03]

Normalized likes:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Normalized comments:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Engagement scores:
[0.70003949 0.70016409 0.70019175 0.70000584 0.70030943 0.70042643
 0.70023755 0.70236186 0.70007079 0.70043034]

Log-scaled views:
[10.79047305 12.19796991 12.35301311  8.99764208 12.82976792 13.14969835
 12.56629629 14.85976486 11.36432104 13.15881873]

90th percentile engagement threshold:
0.7016802820325229

Top 10% mask:
[False False False False False False False  True False F

c:\Users\Muthukumar Ramasamy\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\base.py:689: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)
C:\Users\Muthukumar Ramasamy\AppData\Local\Temp\ipykernel_4116\628737352.py:90: RuntimeWarning: overflow encountered in scalar subtract
  maximum - minimum,


In [9]:
like_ratio[1:2]

array([0.70199879])